In [0]:
! databricks secrets create-scope --scope jdbc-secrets

The Databricks CLI is only supported for interactive use from the web terminal on x86 compute. We recommend using the web terminal to invoke CLI commands. If you would like to interface with Databricks APIs then we recommend using the Databricks Python SDK.


In [0]:
! databricks secrets put --scope jdbc-secrets --key db-host --string-value "adventureworks.postgres.database.azure.com"
! databricks secrets put --scope jdbc-secrets --key db-port --string-value "5432"
! databricks secrets put --scope jdbc-secrets --key db-database --string-value "adventureworks"
! databricks secrets put --scope jdbc-secrets --key db-user --string-value "readonly"
! databricks secrets put --scope jdbc-secrets --key db-password --string-value "Atlas@123"

In [0]:
# Install JDBC Driver: Ensure your Databricks cluster has the PostgreSQL JDBC driver installed. You can add it via the Cluster UI -> Libraries -> Coordinates (org.postgresql:postgresql:42.6.0)

In [0]:
import pyspark.sql.functions as F

# --- Connection Configuration ---
jdbc_hostname = dbutils.secrets.get(scope="jdbc-secrets", key="db-host")
jdbc_port = dbutils.secrets.get(scope="jdbc-secrets", key="db-port")
jdbc_database = dbutils.secrets.get(scope="jdbc-secrets", key="db-database")
jdbc_user = dbutils.secrets.get(scope="jdbc-secrets", key="db-user")
jdbc_password = dbutils.secrets.get(scope="jdbc-secrets", key="db-password")

jdbc_url = f"jdbc:postgresql://{jdbc_hostname}:{jdbc_port}/{jdbc_database}"
connection_properties = {
  "user": jdbc_user,
  "password": jdbc_password,
  "driver": "org.postgresql.Driver"
}

# --- Data Loading ---
# Example: Load SalesOrderHeader table
try:
    # Adjust schema and table name as needed
    sales_order_header_df = spark.read.jdbc(
        url=jdbc_url,
        table="Sales.SalesOrderHeader", # Use actual schema.table
        properties=connection_properties
    )
    print("Successfully loaded data from Sales.SalesOrderHeader")
    display(sales_order_header_df.limit(5))

except Exception as e:
    print(f"Error connecting to or reading from database: {e}")
    dbutils.notebook.exit("Database connection failed")


Successfully loaded data from Sales.SalesOrderHeader


salesorderid,revisionnumber,orderdate,duedate,shipdate,status,onlineorderflag,purchaseordernumber,accountnumber,customerid,salespersonid,territoryid,billtoaddressid,shiptoaddressid,shipmethodid,creditcardid,creditcardapprovalcode,currencyrateid,subtotal,taxamt,freight,totaldue,comment,rowguid,modifieddate
43659,8,2011-05-31T00:00:00Z,2011-06-12T00:00:00Z,2011-06-07T00:00:00Z,5,false,PO522145787,10-4020-000676,29825,279,5,985,985,5,16281,105041Vi84182,null,20565.620600000000000000,1971.514900000000000000,616.098400000000000000,23153.233900000000000000,null,79b65321-39ca-4115-9cba-8fe0903e12e6,2011-06-07T00:00:00Z
43660,8,2011-05-31T00:00:00Z,2011-06-12T00:00:00Z,2011-06-07T00:00:00Z,5,false,PO18850127500,10-4020-000117,29672,279,5,921,921,5,5618,115213Vi29411,null,1294.252900000000000000,124.248300000000000000,38.827600000000000000,1457.328800000000000000,null,738dc42d-d03b-48a1-9822-f95a67ea7389,2011-06-07T00:00:00Z
43661,8,2011-05-31T00:00:00Z,2011-06-12T00:00:00Z,2011-06-07T00:00:00Z,5,false,PO18473189620,10-4020-000442,29734,282,6,517,517,5,1346,85274Vi6854,4,32726.478600000000000000,3153.769600000000000000,985.553000000000000000,36865.801200000000000000,null,d91b9131-18a4-4a11-bc3a-90b6f53e9d74,2011-06-07T00:00:00Z
43662,8,2011-05-31T00:00:00Z,2011-06-12T00:00:00Z,2011-06-07T00:00:00Z,5,false,PO18444174044,10-4020-000227,29994,282,6,482,482,5,10456,125295Vi53935,4,28832.528900000000000000,2775.164600000000000000,867.238900000000000000,32474.932400000000000000,null,4a1ecfc0-cc3a-4740-b028-1c50bb48711c,2011-06-07T00:00:00Z
43663,8,2011-05-31T00:00:00Z,2011-06-12T00:00:00Z,2011-06-07T00:00:00Z,5,false,PO18009186470,10-4020-000510,29565,276,4,1073,1073,5,4322,45303Vi22691,null,419.458900000000000000,40.268100000000000000,12.583800000000000000,472.310800000000000000,null,9b1e7a40-6ae0-4ad3-811c-a64951857c4b,2011-06-07T00:00:00Z


In [0]:
from pyspark.sql.functions import col
# --- Basic Preparation & Selection ---
# Select relevant columns and handle potential data issues (e.g., nulls)
# For simplicity, let's select a few columns and drop rows with null target
# WARNING: This is a very basic example. Real-world prep is more complex.
selected_cols = ["SalesOrderID", "OrderDate", "CustomerID", "SubTotal", "TaxAmt", "Freight", "TotalDue"]
prepared_df = sales_order_header_df.select(*selected_cols).na.drop(subset=["TotalDue"])

# Add a primary key column if not obvious (needed for Feature Store)
# SalesOrderID is likely the primary key here.
prepared_df = prepared_df.withColumn("primary_key", F.col("SalesOrderID"))

prepared_df = prepared_df.withColumn("SubTotal", col("SubTotal").cast("float")) \
                 .withColumn("TaxAmt", col("TaxAmt").cast("float")) \
                 .withColumn("Freight", col("Freight").cast("float")) \
                     .withColumn("TotalDue", col("TotalDue").cast("float")) \

# Persist intermediate results (optional, but good for checkpoints)
prepared_df.write.format("delta").mode("overwrite").save("/mnt/adventureworks/prepared_data2")
# prepared_df = spark.read.format("delta").load("/mnt/adventureworks/prepared_data")

print("Data preparation basic steps completed.")
display(prepared_df.limit(5))

# Pass the prepared DataFrame path or name to the next notebook if needed
# For simplicity, we'll assume subsequent notebooks re-run this or load the saved Delta table.
dbutils.notebook.exit(prepared_df.count()) # Example exit value